<a href="https://colab.research.google.com/github/Not-kh-lily-23/dbank-longitudinal-prediction/blob/main/cohort_tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import os
from google.colab import drive
import re
drive.mount('/content/drive')
base='/content/drive/MyDrive/DementiaBank Project'
csv=os.path.join(base,'pitt_corpus_content.csv')
print("Loading raw parsed transcripts...")
df=pd.read_csv(csv)
df[['participant_id','visit_number']]=df['file_id'].str.extract(r'([A-Za-z0-9]+)-?(\d*)')
df['visit_number']=df['visit_number'].replace('','0').astype(int)
df['group']=df['group'].astype(str).str.strip().str.lower()
df=df.sort_values(by=['participant_id','visit_number'])
bl_df=df.drop_duplicates(subset=['participant_id'],keep='first').copy()
def identify_cohort(group):
    visits=group.sort_values('visit_number')
    first_dx=visits.iloc[0]['group']
    later_dx=visits['group'].values[1:]
    if first_dx in ['control','mci']:
        if 'probablead' in later_dx or 'possiblead' in later_dx:
            return 'Converter'
        elif first_dx=='control' and all(dx=='control' for dx in later_dx):
            return 'Stable Control'
        elif first_dx=='mci' and all(dx=='mci' for dx in later_dx):
            return 'Stable MCI'
    elif first_dx in ['probablead', 'possiblead']:
        return 'Stable AD'
    return 'Exclude'
cohort_map=df.groupby('participant_id').apply(identify_cohort).reset_index(name='cohort_status')
bl_df=df.sort_values(by=['participant_id', 'visit_number']).drop_duplicates(subset=['participant_id'], keep='first').copy()
final_bl_df=pd.merge(bl_df,cohort_map,on='participant_id')
final_bl_df=final_bl_df[final_bl_df['cohort_status']!='Exclude'].copy()
final_bl_df['education']=pd.to_numeric(final_bl_df['education'],errors='coerce')
final_bl_df['education']=final_bl_df.groupby('cohort_status')['education'].transform(lambda x: x.fillna(x.median()))
op=os.path.join(base,'cross_sectional_baseline_master.csv')
def parse_chat_age(age_str):
    if pd.isna(age_str) or not isinstance(age_str,str):
        return None
    parts=re.split(r'[;.]',age_str)
    try:
        years=float(parts[0])
        months=float(parts[1]) if len(parts)>1 and parts[1].isdigit() else 0
        return years+(months/12.0)
    except:
        return None
final_bl_df['age_float']=final_bl_df['age'].apply(parse_chat_age)
final_bl_df['age_float']=final_bl_df.groupby('cohort_status')['age_float'].transform(lambda x: x.fillna(x.median()))
final_bl_df.to_csv(op,index=False)
print("true cohort counts:")
print(final_bl_df['cohort_status'].value_counts())
print(f"missing education after patch: {final_bl_df['education'].isna().sum()}")

Mounted at /content/drive
Loading raw parsed transcripts...


/tmp/ipykernel_2226/1699004055.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cohort_map=df.groupby('participant_id').apply(identify_cohort).reset_index(name='cohort_status')


true cohort counts:
cohort_status
Stable AD         167
Stable Control     98
Stable MCI         16
Converter           1
Name: count, dtype: int64
missing education after patch: 0
